# Reach Start & End Delta -- Violin Plot

## What question this answers

For every algorithm-detected reach that was matched to a ground-truth reach,
how far off were the start and end frames -- and in which direction?

- **start_delta** = `algo_start - gt_start` (negative = algo fires early, positive = late)
- **end_delta** = `algo_end - gt_end` (negative = algo ends early, positive = late)

## What improvement looks like

- Mean and median converge toward 0 (no systematic bias).
- The distribution gets tighter (fewer frames of error).
- Fewer outliers in the tails.
- FP and FN counts drop to 0.

## Red-flag patterns

- **Mean shift away from 0** -- systematic bias (algorithm consistently early or late).
- **Long tails** -- outlier reaches that are very far from GT.
- **Asymmetry between start and end** -- one boundary is much harder than the other.
- **Many FP/FN** -- fundamental count error, not just boundary placement.

In [ ]:
# === PARAMS ===
from pathlib import Path

SNAPSHOT_DIR = Path(r"Y:\2_Connectome\Behavior\MouseReach_Pipeline\Improvement_Snapshots\reach_detection\reach_v7.1.0_visibility_direction_reversal")
FIGSIZE = (12, 5)
DPI = 300
SAVE = False  # Flip to True to write PNG + legend

In [ ]:
# === LOAD ===
import pandas as pd

matches_path = SNAPSHOT_DIR / "metrics" / "reach_matches.csv"
df_all = pd.read_csv(matches_path)

df_matched = df_all[df_all["status"] == "matched"].copy()
df_matched["start_delta"] = df_matched["start_delta"].astype(int)
df_matched["end_delta"] = df_matched["end_delta"].astype(int)

n_fp = int((df_all["status"] == "fp").sum())
n_fn = int((df_all["status"] == "fn").sum())

print(f"Loaded {len(df_all)} rows, {len(df_matched)} matched, {n_fp} FP, {n_fn} FN")

In [ ]:
# === RENDER ===
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from mousereach.improvement.lib.palette import (
    REACH_DETECTION_COLORS, REACH_DETECTION_LABELS, REACH_DETECTION_DELTA_ORDER,
)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, dpi=DPI, sharey=False)

for ax_idx, key in enumerate(REACH_DETECTION_DELTA_ORDER):
    ax = axes[ax_idx]
    data = df_matched[key].values
    color = REACH_DETECTION_COLORS[key]
    label = REACH_DETECTION_LABELS[key]

    # Violin
    if len(data) > 1:
        parts = ax.violinplot(
            data, positions=[0], vert=True, showmeans=True, showmedians=True,
            widths=0.7,
        )
        for pc in parts["bodies"]:
            pc.set_facecolor(color)
            pc.set_alpha(0.5)
            pc.set_edgecolor(color)
        for partname in ("cmeans", "cmedians", "cbars", "cmins", "cmaxes"):
            if partname in parts:
                parts[partname].set_edgecolor(color)
                parts[partname].set_linewidth(1.5)

    # Jittered strip overlay
    jitter = np.random.default_rng(42).uniform(-0.15, 0.15, size=len(data))
    ax.scatter(jitter, data, alpha=0.15, s=6, color=color, zorder=3,
               edgecolors="none")

    # Zero line
    ax.axhline(0, color="#333333", linewidth=1, linestyle="--", alpha=0.6, zorder=1)

    ax.set_xticks([])
    ax.set_ylabel("Delta (frames): algo \u2212 GT", fontsize=11)
    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)

    # Stats annotation
    median_val = float(np.median(data))
    mean_val = float(np.mean(data))
    ax.text(0.02, 0.98, f"median={median_val:.1f}\nmean={mean_val:.1f}\nn={len(data)}",
            transform=ax.transAxes, fontsize=8, verticalalignment="top",
            fontfamily="monospace", color="#555555")

# Footer annotation
fig.text(0.5, -0.02, f"N FP: {n_fp} | N FN: {n_fn}",
         fontsize=10, ha="center", fontfamily="monospace", color="#555555")

fig.suptitle("Reach boundary deltas (matched only)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# === SAVE ===
if SAVE:
    fig_dir = SNAPSHOT_DIR / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)

    png_path = fig_dir / "violin.png"
    fig.savefig(str(png_path), dpi=DPI, bbox_inches="tight", pad_inches=0.15,
                facecolor="white")
    print(f"Saved: {png_path}")

    legend_md = f"""# Reach Boundary Delta -- Violin Plot

## What question this answers

For every algorithm-detected reach that was matched to a ground-truth reach,
how far off were the start and end frames -- and in which direction?

- **start_delta** = `algo_start - gt_start`
- **end_delta** = `algo_end - gt_end`

## What improvement looks like

- Mean and median converge toward 0.
- Distributions get tighter.
- FP and FN counts drop.

## Rendering params

- SNAPSHOT_DIR: `{SNAPSHOT_DIR}`
- FIGSIZE: {FIGSIZE}
- DPI: {DPI}

## Data summary

- Matched reaches: {len(df_matched)}
- FP (phantom): {n_fp}
- FN (miss): {n_fn}
"""
    legend_path = fig_dir / "violin_legend.md"
    legend_path.write_text(legend_md, encoding="utf-8")
    print(f"Saved: {legend_path}")
else:
    print("SAVE=False -- set SAVE=True to write PNG + legend")